In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
%matplotlib inline

In [ ]:
# sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
# cost function
def costFunction(X, y, theta):
    m = len(y)
    h = sigmoid(X.dot(theta))
    error = (y * np.log(h) + (1 - y) * np.log(1 - h))
    cost = -1 / m * np.sum(error)
    grad = 1 / m * X.T.dot(h - y)
    return cost, grad

In [ ]:
# gradient descent
def gradientDescent(X, y, theta, alpha, iterations):
    cost_history = np.zeros(iterations)
    for i in range(iterations):
        cost, grad = costFunction(X, y, theta)
        theta -= alpha * grad
        cost_history[i] = cost
    return theta, cost_history

In [ ]:
# predict
def predict(X, theta):
    return sigmoid(X.dot(theta))

In [ ]:
# accuracy
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

In [ ]:
# =========================
# LOAD DATA
# =========================
df = pd.read_csv('../data/processed_customer_churn.csv')

In [ ]:
# split features and target
X = df.drop('Churn', axis=1)
y = df['Churn'].values  # convert to numpy

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
# =========================
# FEATURE SCALING
# =========================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# =========================
# ADD INTERCEPT
# =========================
X_train = np.c_[np.ones((X_train.shape[0], 1)), X_train]
X_test = np.c_[np.ones((X_test.shape[0], 1)), X_test]

In [ ]:
# =========================
# TRAIN MODEL
# =========================
theta = np.zeros(X_train.shape[1], dtype=float)

theta, cost_history = gradientDescent(X_train, y_train, theta,alpha=0.01,iterations=1000)

In [ ]:
# =========================
# PLOT COST
# =========================
plt.plot(range(1000), cost_history)
plt.xlabel('Iterations')
plt.ylabel('Cost')
plt.title('Cost History')

In [ ]:
# =========================
# TEST PREDICTIONS
# =========================
y_pred = predict(X_test, theta)
y_pred = np.array([1 if i > 0.5 else 0 for i in y_pred])


In [ ]:
# Accuracy
print("Accuracy:", accuracy(y_test, y_pred) * 100)

In [ ]:
# =========================
# NEW DATA PREDICTION
# =========================
df2 = pd.read_csv('../data/new_customers_1.csv')

In [ ]:
# same preprocessing
df2.drop(['Names', 'Location', 'Company', 'Onboard_date'], axis=1, inplace=True)

# scale + intercept
X_new = scaler.transform(df2)
X_new = np.c_[np.ones((X_new.shape[0], 1)), X_new]

# predict
y_pred_new = predict(X_new, theta)
y_pred_new = np.array([1 if i > 0.5 else 0 for i in y_pred_new])

# attach predictions
df2['Churn'] = y_pred_new

df2.head()